# Step 5 — This is RAG

*Step 5 of the AI in Industry lab*

---

## Read this before you run anything

You will join step 4's search to step 1's model: find the right clause, paste it into the prompt, and ask the question.

**What you should end up understanding:** That is all RAG is. Retrieve, stuff into the prompt, instruct the model to stay inside it.

| | |
|---|---|
| **Cost** | 1 API call |
| **Needs earlier steps?** | No. This notebook sets itself up. |
| **Safe to re-run?** | Yes. One API call per run of the answering cell. |

<div style="background:#fff8e1;border-left:6px solid #f9a825;padding:12px 16px;margin:10px 0;border-radius:4px"><b>Uses about 1 API call.</b> Re-running cells is fine, it just uses a little more of your free quota each time.</div>

**Now run the Setup cell.** About 30 seconds. It works even if you skipped every earlier step.

In [ ]:
#@title Setup - run this first (about 30 seconds) { display-mode: "form" }
# Fetches the lab files, installs what is needed, reads your API key.
# Identical in every step notebook, so any step works on its own.
import os, sys, pathlib, subprocess

BASE = "/content" if pathlib.Path("/content").exists() else "."
try:
    os.getcwd()
except OSError:
    os.chdir(BASE)
os.chdir(BASE)

if not pathlib.Path("rough").exists():
    print("Downloading the lab files ...")
    subprocess.run("git clone --depth 1 --quiet "
                   "https://github.com/coolMukul/rough.git rough", shell=True)
os.chdir(f"{BASE}/rough/ai-lab")
sys.path.insert(0, os.getcwd())

print("Installing (slow the first time) ...")
subprocess.run(f"{sys.executable} -m pip install -q -r requirements.txt", shell=True)

try:
    from google.colab import userdata
    os.environ["LLM_API_KEY"] = (userdata.get("LLM_API_KEY") or "").strip()
except Exception:
    pass

if not os.environ.get("LLM_API_KEY"):
    print("\n  NO API KEY. Click the key icon on the left, add a secret named")
    print("  exactly LLM_API_KEY, paste your key from console.groq.com/keys,")
    print("  and turn ON 'Notebook access'. Then run this cell again.")
else:
    print(f"\nReady. Key ending ...{os.environ['LLM_API_KEY'][-4:]}")

Two things you already have:

- **Step 4** finds the right clause
- **Step 1** sends text to a model

Put the clause into the prompt. That is all RAG is.

In [ ]:
from labcore import corpus, tfidf, chat

chunks, texts, _ = corpus()
retrieve = tfidf(texts)

### Your question

In [ ]:
# ===== EDIT ME, then run the cell below =====
question = ("At VFSTR, what is the minimum attendance required in a course, "
            "and what happens if I fall below it?")
k = 4                  # how many clauses to put in the prompt

### Step one: retrieve, and look at what we found

In [ ]:
hits = retrieve(question, k=k)
context = "\n\n".join(f"[{i+1}] {t}" for i, (t, _) in enumerate(hits))

print("THIS IS WHAT WE ARE ABOUT TO PASTE INTO THE PROMPT:")
print()
print(context[:900], "...")

### Step two: the instruction

This is the system prompt. It is doing real work — it is what stops the model wandering off into invention.

In [ ]:
# ===== EDIT ME, then run the cell below =====
system = ("You answer questions about VFSTR academic regulations using ONLY "
          "the context provided. If the context does not contain the answer, "
          'say exactly: "I could not find that in the regulations." '
          "Never use outside knowledge. Quote the rule you relied on.")

### Step three: send both

In [ ]:
answer = chat([
    {"role": "system", "content": system},
    {"role": "user",   "content": f"Context:\n{context}\n\nQuestion: {question}"},
])

print(answer)

**Compare that with step 2.** Same model, same question. The only difference is the paragraph we pasted in first.

---

## Now change it yourself

Two experiments, both worth doing:

**1. Break the system prompt.** Go up to the `system` cell, delete the sentence *"If the context does not contain the answer, say exactly..."*, then ask something that is **not** in the regulations (try *"What is the hostel curfew?"*). Does it start inventing again?

**2. Change `k`.** Set it to `1` and re-run everything below. Then `8`. Watch how much text goes into the prompt, and whether the answer changes.

That second one is the whole of step 6.

---

### Done with step 5

Open the next step's notebook. If something here did not work, **do not stop to debug it** — every step sets itself up from scratch, so the next one will still run.